In [1]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time

In [2]:
def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None, filename_keyword=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")

    if filename_keyword:
        kw = filename_keyword.lower()
        file_paths = [f for f in file_paths if kw in os.path.basename(f).lower()]

    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, engine='calamine')
            df = df.select(pl.all().cast(pl.String))
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0, ignore_errors=True)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)

        # Normalize column names: strip whitespace + BOM
        df.columns = [c.strip().replace('\ufeff', '').replace('\u200b', '') for c in df.columns]

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if df_list:
        return pl.concat(df_list, how='diagonal_relaxed')
    return pl.DataFrame()


def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []
    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))
        df = pl.read_parquet(file).with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)
    if df_list:
        return pl.concat(df_list, how='vertical')
    return pl.DataFrame()


today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

In [3]:
first_glob = os.path.expanduser("~").replace("\\", "/")

folder_paths = {
    "input_performance":           f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_en_performance_global":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_en',
    "input_survey":                f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_t3":                    f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_afcr":                  f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_delayed_closure":       f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
    "mapping_file":                f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc":                   f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_csv_re_direct":         f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")
print("-" * 60)

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_en_performance_global: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_en
input_survey: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_t3: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_afcr: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_delayed_closure: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE
mapping_file: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx
global_hc: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM -

## Survey — New Schema

- `NPS Type` → `_nps_type`, `_promoter`, `_detractor`, `_neutral`, `_survey`
- `DUET Score Type` → `DUET` (Positive=1, Negative=0)
- `Response Text EN` → `_verbatim`

**Removed** (no source in new dump): `_ir`, `_ae`, `_offer`, DUET sub-scores, `delight/usability/ease/trust`

In [4]:
SURVEY_INPUT = input_data(folder_paths["input_survey"], filename_keyword="Survey Dump")

SURVEY_INPUT = SURVEY_INPUT.rename({
    "Conversation_id": "Conversation Id",
    "Agent Email":     "Agent Email ID",
})

SURVEY_INPUT = SURVEY_INPUT.with_columns(
    pl.concat_str(
        [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
        separator="_"
    ).alias("key_survey")
)

survey_nps = (
    SURVEY_INPUT
    .filter(pl.col("NPS Type").is_not_null())
    .with_columns([
        pl.col("NPS Type").alias("_nps_type"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "promoter")
          .then(1).otherwise(0).alias("_promoter"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "detractor")
          .then(1).otherwise(0).alias("_detractor"),
        pl.when(pl.col("NPS Type").str.to_lowercase().is_in(["neutral", "passive"]))
          .then(1).otherwise(0).alias("_neutral"),
        pl.lit(1).alias("_survey"),
    ])
    .select(["key_survey", "Conversation Id", "_nps_type", "_promoter",
             "_detractor", "_neutral", "_survey"])
    .unique()
)

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("DUET Score Type").is_not_null())
    .with_columns(
        pl.when(pl.col("DUET Score Type").str.to_lowercase().str.contains("positive"))
          .then(1).otherwise(0).alias("DUET")
    )
    .select(["key_survey", "Conversation Id", "DUET"])
    .unique()
)

verbatim = (
    SURVEY_INPUT
    .filter(
        pl.col("Response Text EN").is_not_null() &
        (pl.col("Response Text EN").str.strip_chars() != "")
    )
    .with_columns(
        pl.col("Response Text EN")
          .str.replace_all(r"[\r\n\t]+", " ")
          .str.strip_chars()
          .alias("_verbatim")
    )
    .select(["key_survey", "Conversation Id", "_verbatim"])
    .unique()
)

survey_final = (
    survey_nps
    .join(survey_duet, on=["key_survey", "Conversation Id"], how="left")
    .join(verbatim,    on=["key_survey", "Conversation Id"], how="left")
)

print(survey_final.shape)
survey_final.head(3)

(32921, 9)


key_survey,Conversation Id,_nps_type,_promoter,_detractor,_neutral,_survey,DUET,_verbatim
str,str,str,i32,i32,i32,i32,i32,str
"""ankita.giri@concentrix.com_827…","""8271da2d-2661-4475-ad11-a31d58…","""Detractor""",0,1,0,1,0,"""Very bad costumer service"""
"""nungoctrinh.dang@concentrix.co…","""a86facd5-41fd-4359-a3c3-82ae12…","""Detractor""",0,1,0,1,0,"""Last quality service and no an…"
"""yash.trivedi1@concentrix.com_d…","""df418733-9095-4500-9196-18bddb…","""Detractor""",0,1,0,1,0,"""Are you a happy person, fuck o…"


In [5]:
T3_INPUT = input_data(folder_paths["input_t3"], filename_keyword="T3_CNX_AWS")

t3_final = (
    T3_INPUT
    .with_columns(
        pl.when(
            pl.col("Transfer Destination").str.contains("Tier 3", literal=True)
        ).then(1).otherwise(0).alias("T3")
    )
    .filter(pl.col("T3") == 1)
    .with_columns(
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_t3")
    )
    .select(["key_t3", "T3"])
    .unique()
)
t3_final.head(3)

key_t3,T3
str,i32
"""omar.ahmedhassan2@concentrix.c…",1
"""quanghung.nguyen@concentrix.co…",1
"""surya.raut@concentrix.com_e286…",1


In [6]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    all_dataframes = []
    for file_path in glob.glob(os.path.join(folder_path, "*.csv")):
        file_name   = os.path.basename(file_path)
        export_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
        df = pl.read_csv(
            file_path,
            encoding="utf-8",
            schema_overrides={"Itinerary": pl.String},
            infer_schema_length=10000,
            ignore_errors=True
        )
        cast_exprs = []
        for col, dtype in [("Handle Time", pl.Float64), ("Duet", pl.Float64),
                           ("Passed Sessions", pl.Int64), ("Failed Sessions", pl.Int64)]:
            if col in df.columns:
                cast_exprs.append(pl.col(col).cast(dtype, strict=False))
        if cast_exprs:
            df = df.with_columns(cast_exprs)
        df = df.with_columns([
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time')
        ])
        all_dataframes.append(df)
    if not all_dataframes:
        return pl.DataFrame()
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1])).then(1).otherwise(0).alias("Total Sessions")
        ])
        .unique()
    )

In [7]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"], filename_keyword="excess_aws")
    .unique(subset=["user_email", "Conversation ID"], keep="last")
)

delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "user_email", "Conversation ID",
        "Excess Time", "Disconnected Reason (groups)",
        "last_traveler_message_sent_datetime_utc",
    ])
    .rename({
        "user_email":      "Agent Email ID",
        "Conversation ID": "Conversation Id",
        "Excess Time":     "_excess_time_raw",
    })
    .with_columns(
        pl.col("_excess_time_raw").cast(pl.Float64).fill_null(0).alias("Exceed Time"),
    )
    .with_columns([
        (pl.col("Exceed Time") > 0).cast(pl.Int8).alias("Exceed Chat"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("agent").cast(pl.Int8).alias("Agent Disconnect"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("ghost").cast(pl.Int8).alias("Ghost"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("requeue").cast(pl.Int8).alias("Requeued"),
        pl.col("last_traveler_message_sent_datetime_utc")
          .is_null().cast(pl.Int8).alias("Traveler Unresponsive"),
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_delayed_closure"),
    ])
    .drop(["_excess_time_raw", "Disconnected Reason (groups)",
           "last_traveler_message_sent_datetime_utc"])
    .group_by(["Agent Email ID", "Conversation Id", "key_delayed_closure"])
    .agg([
        pl.sum("Exceed Time"),
        pl.sum("Exceed Chat"),
        pl.sum("Agent Disconnect"),
        pl.sum("Ghost"),
        pl.sum("Requeued"),
        pl.sum("Traveler Unresponsive"),
    ])
)

_em = pl.col("Exceed Time") / 60
delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when(_em <= 1).then(pl.lit("01 Mins"))
      .when(_em <= 2).then(pl.lit("02 Mins"))
      .when(_em <= 3).then(pl.lit("03 Mins"))
      .when(_em <= 4).then(pl.lit("04 Mins"))
      .when(_em <= 5).then(pl.lit("05 Mins"))
      .when(_em <= 10).then(pl.lit("05-10 Mins"))
      .when(_em <= 15).then(pl.lit("10-15 Mins"))
      .when(_em <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)
print(delayed_closure.height)

ColumnNotFoundError: "user_email" not found

Resolved plan until failure:

	---> FAILED HERE RESOLVING THIS_NODE <---
DF ["Channel", "Conversation ID", "Initiated Date", "connected_to_agent_datetime_utc"]; PROJECT */18 COLUMNS

In [8]:
RE_DIRECT_INPUT = input_data(folder_paths["input_csv_re_direct"])

re_direct_final = (
    RE_DIRECT_INPUT
    .with_columns(
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                  .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                  .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                  .str.replace_all(r"\s{2,}", " ")
                  .str.strip_chars()
            )
        }
    )
    .with_columns(
        pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_redirect")
    )
    .select(["key_redirect", "Agent People Id", "Re_Direct", "Re-Direct Text"])
    .unique(subset=["key_redirect"], maintain_order=True)
)

In [9]:
PERFORMANCE_INPUT = input_data(
    folder_paths["input_performance"],
    filename_keyword="aws_performance_retail_rawdata"
)

try:
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)

# ── Joined Time parser ─────────────────────────────────────────────────────
def _build_joined_time(time_col: str = "Connected To Agent Time") -> pl.Expr:
    raw = pl.col(time_col).cast(pl.Utf8).str.strip_chars()
    return (
        pl.coalesce([
            raw.str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
            raw.str.strptime(pl.Datetime, "%m/%d/%Y %H:%M",    strict=False),
        ])
        .alias("Joined Time")
    )

# ── Duration parser ────────────────────────────────────────────────────────
def _duration_to_seconds(col: str) -> pl.Expr:
    raw       = pl.col(col).cast(pl.Utf8).str.strip_chars()
    as_number = raw.str.replace_all(",", "").cast(pl.Float64, strict=False)
    h = raw.str.extract(r"^(\d+):\d{2}:\d{2}$", 1).cast(pl.Float64, strict=False)
    m = raw.str.extract(r"^\d+:(\d{2}):\d{2}$", 1).cast(pl.Float64, strict=False)
    s = raw.str.extract(r"^\d+:\d{2}:(\d{2})$", 1).cast(pl.Float64, strict=False)
    return pl.when(as_number.is_not_null()).then(as_number).otherwise(h*3600 + m*60 + s).alias(col)

# ── AWS column adapter ─────────────────────────────────────────────────────
PERFORMANCE_INPUT = (
    PERFORMANCE_INPUT
    .with_columns(_build_joined_time())
    .rename({
        "Handle Time":                    "Handle Time (Sum)",
        "Talk Time":                      "Talk Time (Sum)",
        "Acw Duration":                   "Wrap Up Time (Sum)",
        "Agent Vendor Location":          "Agent Business Location",
        "Outbound Initiated (Yes / No)":  "Initiated Outbound (Yes / No)",
        "Product":                        "Latest VA Product",
        "Intent":                         "Latest VA Intent",
        "Locale":                         "Language",
    })
    .with_columns([
        pl.col("Latest VA Product").fill_null("UNKNOWN").alias("Latest VA Product"),
        pl.col("Latest VA Intent").fill_null("UNKNOWN").alias("Latest VA Intent"),
    ])
)

print(PERFORMANCE_INPUT.select(["Connected To Agent Time", "Joined Time"]).head(5))

existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Handle Time (Sum)":  pl.Float64,
    "Talk Time (Sum)":    pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Hold Time (Sum)":    pl.Float64,
    "Handle (Count)":     pl.Int64,
}

casts = []
for _col, _dtype in columns_to_cast.items():
    if _col not in existing_cols:
        continue
    if _dtype == pl.Float64:
        casts.append(_duration_to_seconds(_col))
    else:
        casts.append(pl.col(_col).cast(_dtype, strict=False).alias(_col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)

# ── Routing Profile → LOB + Agent Queue Group Name override ───────────────
LG_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Car_Activity",
    "Chat_AC_GLB_EN_Lodging_Nesting",
    "Chat_AC_GLB_EN_Lodging_Proficient",
]
NL_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Proficient",
    "Chat_AC_GLB_EN_NL_Nesting",
]

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Agent Routing Profile Name").alias("Agent Queue Group Name"),
    pl.when(pl.col("Agent Routing Profile Name").is_in(LG_CHAT_PROFILES))
      .then(pl.lit("LG_Chat"))
      .when(pl.col("Agent Routing Profile Name").is_in(NL_CHAT_PROFILES))
      .then(pl.lit("NL Chat"))
      .otherwise(pl.col("Agent Routing Profile Name"))
      .alias("LOB"),
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date"),
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)"),
])

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Vendor Location', 'Outbound Initiated (Yes / No)', 'Business Segment Name', 'Partner Name', 'Locale', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time', 'Talk Time', 'Assigned Agent Time', 'Acw Duration', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Intent', 'Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time']
shape: (5, 2)
┌─────────────────────────┬─────────────────────┐
│ Connected To Agent Time ┆ Joined Time         │
│ ---                     ┆ ---                 │
│ str                     ┆ datetime[μs]        │
╞════════════════

In [10]:
GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"]).unique(subset=["SSO ID"], keep="last")

merged_global_hc = PERFORMANCE_NEXT_STEP.join(
    GLOBAL_HC[['SSO ID', "Name", "Supervisor", "Production Start date", "Agent/Non Agent"]],
    left_on=["Agent Email ID"],
    right_on=['SSO ID'],
    how='left'
)

mapping          = pl.read_excel(folder_paths["mapping_file"], sheet_name="queue_name_mapping")
mapping_harmony  = pl.read_excel(folder_paths["mapping_file"], sheet_name="harmony_agents")
mapping_lc_scheme= pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")
mapping_bq_list  = (
    pl.read_excel(folder_paths["mapping_file"], sheet_name="bq_list")
    .filter(pl.col("Sub") == "en")
    .select([pl.col("Agent Email ID").cast(pl.String), pl.col("Quartile Flag").cast(pl.String)])
)

performance_cleaned = merged_global_hc.join(mapping, on="Agent Queue Group Name", how='left')
performance_cleaned = (
    performance_cleaned
    .join(mapping_harmony, on="Agent Email ID", how="left")
    .with_columns(pl.col("Harmony Flag").fill_null("Non Harmony"))
)
performance_cleaned = performance_cleaned.join(mapping_bq_list, on="Agent Email ID", how="left")

lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")

Could not determine dtype for column 5, falling back to string


In [11]:
# NOTE: _promoter/_detractor/_neutral/_survey/_nps_type now come from survey_final join.
# CCR72/_fup_72/_rr/_offer/_ir/_ae removed — no source columns in new schema.
performance_processed = performance_cleaned.with_columns([
    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.strftime("%G_%V").str.slice(2).alias("_PST.WeekNo"),
    (
        pl.col("Joined Time") -
        pl.col("Joined Time").dt.weekday() * pl.duration(days=1)
    ).dt.strftime("WB_%y/%m/%d").alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),

    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null("")
    ], separator="_").alias("_key"),

    # Pre/Post flag: unchanged from original
    pl.when(pl.col("Joined Time") <= pl.datetime(2025, 7, 7))
      .then(pl.lit("Pre")).otherwise(pl.lit("Post")).alias("Pre/Post"),
])

# AON Days + Status
performance_processed = performance_processed.with_columns(
    (
        (pl.col("_PST.Date").cast(pl.Date) - pl.col("Production Start date").cast(pl.Date))
        / pl.duration(days=1)
    ).cast(pl.Int32).alias("AON_Days")
)
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() &
        (pl.col("Agent/Non Agent").is_in(["Agent","ID Deleted"]))
    ).then(pl.lit("Nesting"))
      .when(pl.col("AON_Days") > 180).then(pl.lit("> 180 Days"))
      .when(pl.col("AON_Days") >= 91).then(pl.lit("91 - 180"))
      .when(pl.col("AON_Days") >= 61).then(pl.lit("61 - 90"))
      .when(pl.col("AON_Days") >= 31).then(pl.lit("31 - 60"))
      .when(pl.col("AON_Days") >= 0).then(pl.lit("00 - 30"))
      .otherwise(None).alias("AON Status")
])

# LC threshold (join_asof)
performance_processed = performance_processed.join_asof(
    mapping_lc_scheme,
    left_on="_PST.Date",
    right_on="Effective Date",
    by="LOB",
    strategy="backward"
)
performance_processed = performance_processed.with_columns([
    (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc"),
    (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat"),
])

print(performance_processed.select(["_PST.Month","Threshole_LC","Handle Time (Sum)","_lc"])
      .filter(pl.col("_lc") == 1).head(3))

# Composite keys
performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY"),
    pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("EmailID_ConversationID_KEY"),
    pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("PeopleID_ConversationID_KEY"),
])

# 30-min PST interval (native, no map_elements)
_pst_hour = pl.col("Joined Time").dt.hour()
_pst_min  = pl.col("Joined Time").dt.minute()
_sm = pl.when(_pst_min < 30).then(pl.lit(0)).otherwise(pl.lit(30))
_em = pl.when(_pst_min < 30).then(pl.lit(29)).otherwise(pl.lit(59))
performance_processed = performance_processed.with_columns(
    pl.concat_str([
        _pst_hour.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        _sm.cast(pl.Utf8).str.zfill(2), pl.lit("-"),
        _pst_hour.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        _em.cast(pl.Utf8).str.zfill(2),
    ]).alias("_PST.Interval")
)

print(performance_processed.columns)

shape: (3, 4)
┌────────────┬──────────────┬───────────────────┬─────┐
│ _PST.Month ┆ Threshole_LC ┆ Handle Time (Sum) ┆ _lc │
│ ---        ┆ ---          ┆ ---               ┆ --- │
│ str        ┆ i64          ┆ f64               ┆ i8  │
╞════════════╪══════════════╪═══════════════════╪═════╡
│ 26_06      ┆ 2400         ┆ 2556.0            ┆ 1   │
│ 26_06      ┆ 2400         ┆ 2706.0            ┆ 1   │
│ 26_06      ┆ 2400         ┆ 3199.0            ┆ 1   │
└────────────┴──────────────┴───────────────────┴─────┘
['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Business Segment Name', 'Partner Name', 'Language', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time (Sum)', 'Talk Time (Sum)', 'Assigned Agent Time', 'Wrap Up Time (Sum)', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disc

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5628\1919043134.py:52: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  performance_processed = performance_processed.join_asof(


In [12]:
performance_merged_survey_t3 = (
    performance_processed
    .join(t3_final,                                    left_on="_conver_unique",              right_on="key_t3",               how="left")
    .join(delayed_closure,                             left_on="EmailID_ConversationID_KEY",  right_on="key_delayed_closure",  how="left")
    .join(re_direct_final,                             left_on="PeopleID_ConversationID_KEY", right_on="key_redirect",         how="left")
    .join(survey_final.drop("Conversation Id"),        left_on="EmailID_ConversationID_KEY",  right_on="key_survey",           how="left")
    .join(afcr_input,                                  on=["Agent Email ID", "Conversation Id"],                               how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date","Production Start date","AON_Days"]).head())

selected_columns = [
    "_PST.Date","_PST.Year","_PST.WeekNo","_PST.Week","_PST.Month","_PST.Interval",
    "Agent People Id","Agent Email ID","Conversation Id","Joined Time",
    "Agent Queue Group Name","Supervisor","Agent Business Location","AON Status","Name","Language","LOB",
    "Latest VA Intent","Latest VA Product","Business Segment Name","Partner Name",
    "Handle (Count)","Handle Time (Sum)","Response Count","Response Time","Hold Time (Sum)",
    "Wrap Up Time (Sum)","Talk Time (Sum)","_aob","_survey","_promoter",
    "_detractor","_neutral","_nps_type","_lc",
    "T3","_verbatim","Agent/Non Agent","Harmony Flag",
    "Pre/Post","_conver_unique","Re_Direct","Re-Direct Text",
    "Exceed Time","Exceed Chat","Agent Disconnect",
    "Ghost","Requeued","Exceed Bucket","DUET",
    "Traveler Unresponsive","Short Chat","Passed Sessions","Failed Sessions","Total Sessions",
    "Quartile Flag",
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Missing columns:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns).unique()
performance_filtered = performance_filtered.sort(
    by=["Conversation Id","Agent Email ID","Joined Time"],
    descending=[False, False, True]
).with_columns(
    (pl.col("Joined Time").cum_count().over(["Conversation Id","Agent Email ID"]) > 1)
    .cast(pl.Int8).alias("Duplicate_Flag")
)
performance_filtered.head(3)

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Business Segment Name', 'Partner Name', 'Language', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time (Sum)', 'Talk Time (Sum)', 'Assigned Agent Time', 'Wrap Up Time (Sum)', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Latest VA Intent', 'Latest VA Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time', 'Joined Time', 'LOB', 'Joined Date', 'Join Time (VNT)', 'Join Date (VNT)', 'Name', 'Supervisor', 'Production Start date', 'Agent/Non Agent', 'MainLOBs', 'CFG Groups', 'LOB_right', 'Forecast

_PST.Date,_PST.Year,_PST.WeekNo,_PST.Week,_PST.Month,_PST.Interval,Agent People Id,Agent Email ID,Conversation Id,Joined Time,Agent Queue Group Name,Supervisor,Agent Business Location,AON Status,Name,Language,LOB,Latest VA Intent,Latest VA Product,Business Segment Name,Partner Name,Handle (Count),Handle Time (Sum),Response Count,Response Time,Hold Time (Sum),Wrap Up Time (Sum),Talk Time (Sum),_aob,_survey,_promoter,_detractor,_neutral,_nps_type,_lc,T3,_verbatim,Agent/Non Agent,Harmony Flag,Pre/Post,_conver_unique,Re_Direct,Re-Direct Text,Exceed Time,Exceed Chat,Agent Disconnect,Ghost,Requeued,Exceed Bucket,DUET,Traveler Unresponsive,Short Chat,Passed Sessions,Failed Sessions,Total Sessions,Quartile Flag,Duplicate_Flag
date,i32,str,str,str,str,str,str,str,datetime[μs],str,str,str,str,str,str,str,str,str,str,str,i64,f64,str,str,f64,f64,f64,i32,i32,i32,i32,i32,str,i8,i32,str,str,str,str,str,i32,str,f64,i64,i64,i64,i64,str,i32,i64,i8,i64,i64,i32,str,i8
2026-06-27,2026,"""26_26""","""WB_26/06/21""","""26_06""","""12:30-12:59""","""525423869""","""raisa.sultana@concentrix.com""","""000013b6-b16c-4542-bda1-615a87…",2026-06-27 12:41:55,"""Voice_AC_GLB_EN_Lodging_Profic…","""Meghoranjani Das ""","""Concentrix (Kolkata)""","""Nesting""","""Raisa Sultana""","""en_US""","""Voice_AC_GLB_EN_Lodging_Profic…","""CANCEL""","""LODGING""","""Expedia United States""","""Expedia""",1,982.0,null,null,509.0,5.0,468.0,1,null,null,null,null,null,null,null,null,"""Agent""","""Non Harmony""","""Post""","""raisa.sultana@concentrix.com_0…",null,null,null,null,null,null,null,null,null,null,0,null,null,null,null,0
2026-06-15,2026,"""26_25""","""WB_26/06/14""","""26_06""","""06:00-06:29""","""461958563""","""saif.mohamed@concentrix.com""","""00004120-bb1c-441f-94ee-8557ef…",2026-06-15 06:21:22,"""Chat_AC_GLB_EN_Proficient""","""Mohamed Ibrahim Ahmed ""","""Concentrix (Cairo)""","""> 180 Days""","""Saif Mahmoud Abdelaziz Mohamed""","""en_US""","""NL Chat""","""UNKNOWN""","""UNKNOWN""","""Expedia United States""","""Expedia""",1,522.0,"""7""","""167""",0.0,16.0,null,0,null,null,null,null,null,0,null,null,"""Agent""","""Non Harmony""","""Post""","""saif.mohamed@concentrix.com_00…",null,null,null,null,null,null,null,null,null,null,0,null,null,null,"""MQ2""",0
2026-06-15,2026,"""26_25""","""WB_26/06/14""","""26_06""","""09:00-09:29""","""39218014""","""thuytram.nguyen@concentrix.com""","""0000b958-b235-4255-81d8-d5f27b…",2026-06-15 09:13:56,"""Chat_AC_GLB_EN_Car_Activity""","""Truong Thien Thanh Toan ""","""Concentrix (Ho Chi Minh City)""","""31 - 60""","""Nguyen Thuy Tram""","""en_US""","""LG_Chat""","""UNKNOWN""","""UNKNOWN""","""Hotels United States""","""Hotels.com""",1,1598.0,"""5""","""519""",0.0,16.0,null,0,null,null,null,null,null,null,null,null,"""Agent""","""Non Harmony""","""Post""","""thuytram.nguyen@concentrix.com…",null,null,null,null,null,null,null,null,null,null,0,1,0,1,null,0


In [13]:
removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

def clean_id(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.replace_all(r'[""\u201c\u201d]', "")
            .str.strip_chars()
    )

if "Conversation Id" not in removed_id.columns:
    raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

removed_id_norm = (
    removed_id
    .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .drop_nulls()
    .filter(pl.col("Conversation Id") != "")
    .unique()
)

performance_filtered = (
    performance_filtered
    .with_columns(clean_id(pl.col("Conversation Id")).alias("__conv_norm"))
    .join(
        removed_id_norm
            .select(pl.col("Conversation Id").alias("__conv_norm"))
            .with_columns(pl.lit(1).alias("__rm")),
        on="__conv_norm",
        how="left",
    )
    .with_columns(pl.col("__rm").fill_null(0).alias("IDs Removed"))
    .drop(["__conv_norm","__rm"])
)

flagged = performance_filtered.select(pl.col("IDs Removed").sum().alias("flagged")).item()
total   = performance_filtered.height
print(f"Flagged (IDs Removed=1): {flagged} / {total} rows")

Flagged (IDs Removed=1): 0 / 363944 rows


In [ ]:
# region EXPORT
performance_unique   = performance_filtered.unique()
performance_all_site = performance_unique

output_dir = folder_paths["output_en_performance_global"]
os.makedirs(output_dir, exist_ok=True)

for (month_value,), group in performance_all_site.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))
    group.write_parquet(os.path.join(output_dir, f"{base_name}_performance_en_global.parquet"))

# Merge tất cả tháng với diagonal_relaxed (backward compat với tháng cũ có thêm cột)
parquet_files = glob.glob(os.path.join(output_dir, "*_performance_en_global.parquet"))
out_path_total = os.path.join(output_dir, "all_months_performance_en_global.parquet")

if parquet_files:
    lazy_frames = [pl.scan_parquet(f) for f in sorted(parquet_files)]
    (
        pl.concat(lazy_frames, how="diagonal_relaxed")
        .collect()
        .write_parquet(out_path_total)
    )
    print(f"Merged {len(parquet_files)} months → {out_path_total}")
# endregion

In [ ]:
print(performance_all_site['_PST.Month'].drop_nulls().unique().sort())

In [ ]:
file_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_en/all_months_performance_en_global.parquet'

performance_df = pl.scan_parquet(file_path).collect()

aht_summary = (
    performance_df
    .filter(pl.col("Duplicate_Flag").fill_null(0) == 0)
    .with_columns([
        pl.col("_PST.Date").cast(pl.Date).dt.truncate("1q").alias("Effective_Date"),
        pl.concat_str([
            pl.col("_PST.Date").cast(pl.Date).dt.year().cast(pl.String),
            pl.lit("-Q"),
            pl.col("_PST.Date").cast(pl.Date).dt.quarter().cast(pl.String)
        ]).alias("Year_Quarter")
    ])
    .group_by(["LOB","Year_Quarter","Effective_Date"])
    .agg([
        (pl.col("Handle Time (Sum)").sum() / pl.col("Handle (Count)").sum()).alias("AHT_Actual"),
        pl.col("Conversation Id").n_unique().alias("Total_Conversations")
    ])
    .sort(["LOB","Year_Quarter"])
    .with_columns([
        pl.col("AHT_Actual").shift(1).over("LOB").alias("Prev_Actual_AHT")
    ])
    .with_columns([
        pl.when(pl.col("Prev_Actual_AHT").is_not_null())
          .then((pl.col("Prev_Actual_AHT") * 0.95).round(0).cast(pl.Int64))
          .otherwise(None).alias("Target AHT")
    ])
    .filter(pl.col("Target AHT").is_not_null())
    .sort(["Year_Quarter","LOB"])
)

with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(aht_summary)

In [ ]:
quartile_df = (
    performance_all_site
    .group_by(['_PST.Month','_PST.WeekNo','Agent Business Location',
               'Agent Email ID','Agent Queue Group Name','Latest VA Product'])
    .agg([
        pl.col('_promoter').sum().alias('Promoter'),
        pl.col('_detractor').sum().alias('Detractor'),
        pl.col('_survey').sum().alias('Survey')
    ])
    .with_columns([
        pl.when(pl.col('Survey').is_null() | (pl.col('Survey') == 0))
          .then(None)
          .otherwise(((pl.col('Promoter') - pl.col('Detractor')) / pl.col('Survey') * 100))
          .alias('NPS')
    ])
)

quartile_df = (
    quartile_df
    .with_columns([
        pl.when(pl.col('NPS').is_null()).then(None)
          .when(pl.col('NPS') >= 75).then(pl.lit('Quartile 1'))
          .when(pl.col('NPS') >= 50).then(pl.lit('Quartile 2'))
          .when(pl.col('NPS') >= 25).then(pl.lit('Quartile 3'))
          .when(pl.col('NPS') < 25).then(pl.lit('Quartile 4'))
          .otherwise(None)
          .alias('Quartile')
    ])
)

quartile_df.write_excel('quartile2.xlsx')
print(quartile_df.shape)